In [1]:
# %pip install duckdb pyarrow -q

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:,.6f}".format)

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, ID_COL

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

duckdb_temp_dir = PROCESSED_DATA_DIR / "_duckdb_temp"
duckdb_temp_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"PRAGMA temp_directory='{duckdb_temp_dir.as_posix()}';")
con.execute("PRAGMA threads=4;")

installments_path = (RAW_DATA_DIR / "installments_payments.csv").as_posix()
pos_path = (RAW_DATA_DIR / "POS_CASH_balance.csv").as_posix()
cc_path = (RAW_DATA_DIR / "credit_card_balance.csv").as_posix()

print("DuckDB ready")
print("Installments path:", installments_path)
print("POS path:", pos_path)
print("CC path:", cc_path)

DuckDB ready
Installments path: C:/Coding/Home-Credit-Default-Risk/data/raw/installments_payments.csv
POS path: C:/Coding/Home-Credit-Default-Risk/data/raw/POS_CASH_balance.csv
CC path: C:/Coding/Home-Credit-Default-Risk/data/raw/credit_card_balance.csv


In [2]:
train_ids = pd.read_csv(RAW_DATA_DIR / "application_train.csv", usecols=[ID_COL])
test_ids = pd.read_csv(RAW_DATA_DIR / "application_test.csv", usecols=[ID_COL])

base = pd.concat([train_ids, test_ids], axis=0, ignore_index=True)
base = base.drop_duplicates(subset=[ID_COL]).reset_index(drop=True)

print("Base shape:", base.shape)
display(base.head())

Base shape: (356255, 1)


,SK_ID_CURR
0,100002
1,100003
2,100004
3,100006
4,100007


In [3]:
con.execute(f"""
COPY (
    WITH src AS (
        SELECT
            SK_ID_CURR,
            SK_ID_PREV,
            NUM_INSTALMENT_VERSION,
            NUM_INSTALMENT_NUMBER,
            DAYS_INSTALMENT,
            DAYS_ENTRY_PAYMENT,
            AMT_INSTALMENT,
            AMT_PAYMENT,
            CASE
                WHEN AMT_INSTALMENT IS NULL OR AMT_INSTALMENT = 0 THEN NULL
                ELSE AMT_PAYMENT / AMT_INSTALMENT
            END AS INS_PAYMENT_RATIO,
            AMT_PAYMENT - AMT_INSTALMENT AS INS_PAYMENT_DIFF,
            GREATEST(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT, 0) AS INS_DPD,
            GREATEST(DAYS_INSTALMENT - DAYS_ENTRY_PAYMENT, 0) AS INS_DBD,
            CASE WHEN DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT > 0 THEN 1 ELSE 0 END AS INS_IS_LATE,
            CASE WHEN DAYS_INSTALMENT - DAYS_ENTRY_PAYMENT > 0 THEN 1 ELSE 0 END AS INS_IS_EARLY,
            CASE WHEN AMT_PAYMENT < AMT_INSTALMENT THEN 1 ELSE 0 END AS INS_IS_UNDERPAID,
            CASE WHEN AMT_PAYMENT > AMT_INSTALMENT THEN 1 ELSE 0 END AS INS_IS_OVERPAID
        FROM read_csv_auto('{installments_path}', header=true)
    )
    SELECT
        SK_ID_CURR,
        COUNT(DISTINCT SK_ID_PREV) AS INS_SK_ID_PREV_NUNIQUE,
        COUNT(*) AS INS_RECORD_COUNT,
        COUNT(DISTINCT NUM_INSTALMENT_VERSION) AS INS_NUM_INSTALMENT_VERSION_NUNIQUE,
        MAX(NUM_INSTALMENT_NUMBER) AS INS_NUM_INSTALMENT_NUMBER_MAX,

        AVG(INS_DPD) AS INS_DPD_MEAN,
        MAX(INS_DPD) AS INS_DPD_MAX,
        SUM(INS_DPD) AS INS_DPD_SUM,

        AVG(INS_DBD) AS INS_DBD_MEAN,
        MAX(INS_DBD) AS INS_DBD_MAX,
        SUM(INS_DBD) AS INS_DBD_SUM,

        SUM(INS_IS_LATE) AS INS_LATE_COUNT,
        AVG(INS_IS_LATE) AS INS_LATE_RATIO,
        SUM(INS_IS_EARLY) AS INS_EARLY_COUNT,
        AVG(INS_IS_EARLY) AS INS_EARLY_RATIO,

        SUM(INS_IS_UNDERPAID) AS INS_UNDERPAID_COUNT,
        AVG(INS_IS_UNDERPAID) AS INS_UNDERPAID_RATIO,
        SUM(INS_IS_OVERPAID) AS INS_OVERPAID_COUNT,
        AVG(INS_IS_OVERPAID) AS INS_OVERPAID_RATIO,

        AVG(INS_PAYMENT_RATIO) AS INS_PAYMENT_RATIO_MEAN,
        MIN(INS_PAYMENT_RATIO) AS INS_PAYMENT_RATIO_MIN,
        MAX(INS_PAYMENT_RATIO) AS INS_PAYMENT_RATIO_MAX,

        AVG(INS_PAYMENT_DIFF) AS INS_PAYMENT_DIFF_MEAN,
        MIN(INS_PAYMENT_DIFF) AS INS_PAYMENT_DIFF_MIN,
        MAX(INS_PAYMENT_DIFF) AS INS_PAYMENT_DIFF_MAX,
        SUM(INS_PAYMENT_DIFF) AS INS_PAYMENT_DIFF_SUM,

        SUM(AMT_PAYMENT) AS INS_AMT_PAYMENT_SUM,
        AVG(AMT_PAYMENT) AS INS_AMT_PAYMENT_MEAN,
        SUM(AMT_INSTALMENT) AS INS_AMT_INSTALMENT_SUM,
        AVG(AMT_INSTALMENT) AS INS_AMT_INSTALMENT_MEAN,

        MIN(DAYS_INSTALMENT) AS INS_DAYS_INSTALMENT_MIN,
        MAX(DAYS_INSTALMENT) AS INS_DAYS_INSTALMENT_MAX,
        MIN(DAYS_ENTRY_PAYMENT) AS INS_DAYS_ENTRY_PAYMENT_MIN,
        MAX(DAYS_ENTRY_PAYMENT) AS INS_DAYS_ENTRY_PAYMENT_MAX,

        CASE
            WHEN SUM(AMT_INSTALMENT) IS NULL OR SUM(AMT_INSTALMENT) = 0 THEN NULL
            ELSE SUM(AMT_PAYMENT) / SUM(AMT_INSTALMENT)
        END AS INS_TOTAL_PAYMENT_TO_INSTALMENT_RATIO
    FROM src
    GROUP BY SK_ID_CURR
) TO '{(PROCESSED_DATA_DIR / "installments_features.parquet").as_posix()}'
(FORMAT PARQUET);
""")

print("Saved installments_features.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved installments_features.parquet


In [4]:
installments_features = pd.read_parquet(PROCESSED_DATA_DIR / "installments_features.parquet")
print("Installments features shape:", installments_features.shape)
display(installments_features.head())

Installments features shape: (339587, 35)


,SK_ID_CURR,INS_SK_ID_PREV_NUNIQUE,INS_RECORD_COUNT,INS_NUM_INSTALMENT_VERSION_NUNIQUE,INS_NUM_INSTALMENT_NUMBER_MAX,INS_DPD_MEAN,INS_DPD_MAX,INS_DPD_SUM,INS_DBD_MEAN,INS_DBD_MAX,INS_DBD_SUM,INS_LATE_COUNT,INS_LATE_RATIO,INS_EARLY_COUNT,INS_EARLY_RATIO,INS_UNDERPAID_COUNT,INS_UNDERPAID_RATIO,INS_OVERPAID_COUNT,INS_OVERPAID_RATIO,INS_PAYMENT_RATIO_MEAN,INS_PAYMENT_RATIO_MIN,INS_PAYMENT_RATIO_MAX,INS_PAYMENT_DIFF_MEAN,INS_PAYMENT_DIFF_MIN,INS_PAYMENT_DIFF_MAX,INS_PAYMENT_DIFF_SUM,INS_AMT_PAYMENT_SUM,INS_AMT_PAYMENT_MEAN,INS_AMT_INSTALMENT_SUM,INS_AMT_INSTALMENT_MEAN,INS_DAYS_INSTALMENT_MIN,INS_DAYS_INSTALMENT_MAX,INS_DAYS_ENTRY_PAYMENT_MIN,INS_DAYS_ENTRY_PAYMENT_MAX,INS_TOTAL_PAYMENT_TO_INSTALMENT_RATIO
0,145280,6,105,3,48,0.104762,4.000000,11.000000,6.695238,30.000000,703.000000,3.000000,0.028571,59.000000,0.561905,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,"839,176.785000","7,992.159857","839,176.785000","7,992.159857","-2,894.000000",-8.000000,"-2,903.000000",-12.000000,1.000000
1,198934,8,69,2,18,0.057971,3.000000,4.000000,13.913043,60.000000,960.000000,2.000000,0.028986,64.000000,0.927536,2.000000,0.028986,0.000000,0.000000,0.985507,0.059746,1.000000,-592.235870,"-38,422.800000",0.000000,"-40,864.275000","1,287,183.060000","18,654.826957","1,328,047.335000","19,247.062826","-1,810.000000",-6.000000,"-1,817.000000",-18.000000,0.969230
2,158712,1,14,1,12,2.285714,17.000000,32.000000,10.142857,15.000000,142.000000,2.000000,0.142857,12.000000,0.857143,4.000000,0.285714,0.000000,0.000000,0.857143,0.000023,1.000000,-838.202143,"-5,867.280000",0.000000,"-11,734.830000","70,404.345000","5,028.881786","82,139.175000","5,867.083929",-498.000000,-168.000000,-510.000000,-179.000000,0.857135
3,191933,3,160,2,106,1.893750,30.000000,303.000000,4.843750,33.000000,775.000000,19.000000,0.118750,64.000000,0.400000,36.000000,0.225000,0.000000,0.000000,0.887500,0.000050,1.000000,-808.478156,"-22,292.685000",0.000000,"-129,356.505000","737,816.130000","4,611.350812","867,172.635000","5,419.828969","-2,919.000000",-15.000000,"-2,919.000000",-15.000000,0.850830
4,144004,2,10,2,6,0.000000,0.000000,0.000000,12.300000,26.000000,123.000000,0.000000,0.000000,10.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,"313,822.395000","31,382.239500","313,822.395000","31,382.239500","-1,003.000000",-551.000000,"-1,012.000000",-565.000000,1.000000


In [5]:
con.execute(f"""
COPY (
    SELECT
        SK_ID_CURR,
        COUNT(DISTINCT SK_ID_PREV) AS POS_SK_ID_PREV_NUNIQUE,
        COUNT(*) AS POS_RECORD_COUNT,

        MIN(MONTHS_BALANCE) AS POS_MONTHS_BALANCE_MIN,
        MAX(MONTHS_BALANCE) AS POS_MONTHS_BALANCE_MAX,
        COUNT(MONTHS_BALANCE) AS POS_MONTHS_BALANCE_SIZE,

        AVG(CNT_INSTALMENT) AS POS_CNT_INSTALMENT_MEAN,
        MAX(CNT_INSTALMENT) AS POS_CNT_INSTALMENT_MAX,
        AVG(CNT_INSTALMENT_FUTURE) AS POS_CNT_INSTALMENT_FUTURE_MEAN,
        MAX(CNT_INSTALMENT_FUTURE) AS POS_CNT_INSTALMENT_FUTURE_MAX,

        AVG(SK_DPD) AS POS_SK_DPD_MEAN,
        MAX(SK_DPD) AS POS_SK_DPD_MAX,
        AVG(SK_DPD_DEF) AS POS_SK_DPD_DEF_MEAN,
        MAX(SK_DPD_DEF) AS POS_SK_DPD_DEF_MAX,

        SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END) AS POS_DPD_COUNT,
        AVG(CASE WHEN SK_DPD > 0 THEN 1.0 ELSE 0.0 END) AS POS_DPD_RATIO,

        SUM(CASE WHEN SK_DPD_DEF > 0 THEN 1 ELSE 0 END) AS POS_DPD_DEF_COUNT,
        AVG(CASE WHEN SK_DPD_DEF > 0 THEN 1.0 ELSE 0.0 END) AS POS_DPD_DEF_RATIO,

        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Active' THEN 1 ELSE 0 END) AS POS_STATUS_ACTIVE_COUNT,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Completed' THEN 1 ELSE 0 END) AS POS_STATUS_COMPLETED_COUNT,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Signed' THEN 1 ELSE 0 END) AS POS_STATUS_SIGNED_COUNT,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Demand' THEN 1 ELSE 0 END) AS POS_STATUS_DEMAND_COUNT,

        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Active' THEN 1.0 ELSE 0.0 END) AS POS_STATUS_ACTIVE_RATIO,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Completed' THEN 1.0 ELSE 0.0 END) AS POS_STATUS_COMPLETED_RATIO,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Signed' THEN 1.0 ELSE 0.0 END) AS POS_STATUS_SIGNED_RATIO,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Demand' THEN 1.0 ELSE 0.0 END) AS POS_STATUS_DEMAND_RATIO
    FROM read_csv_auto('{pos_path}', header=true)
    GROUP BY SK_ID_CURR
) TO '{(PROCESSED_DATA_DIR / "pos_cash_features.parquet").as_posix()}'
(FORMAT PARQUET);
""")

print("Saved pos_cash_features.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved pos_cash_features.parquet


In [6]:
pos_features = pd.read_parquet(PROCESSED_DATA_DIR / "pos_cash_features.parquet")
print("POS features shape:", pos_features.shape)
display(pos_features.head())

POS features shape: (337252, 26)


,SK_ID_CURR,POS_SK_ID_PREV_NUNIQUE,POS_RECORD_COUNT,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_MONTHS_BALANCE_SIZE,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX,POS_DPD_COUNT,POS_DPD_RATIO,POS_DPD_DEF_COUNT,POS_DPD_DEF_RATIO,POS_STATUS_ACTIVE_COUNT,POS_STATUS_COMPLETED_COUNT,POS_STATUS_SIGNED_COUNT,POS_STATUS_DEMAND_COUNT,POS_STATUS_ACTIVE_RATIO,POS_STATUS_COMPLETED_RATIO,POS_STATUS_SIGNED_RATIO,POS_STATUS_DEMAND_RATIO
0,121756,7,87,-58,-2,87,17.344828,24.000000,11.091954,24.000000,0.000000,0,0.000000,0,0.000000,0.000000,0.000000,0.000000,82.000000,5.000000,0.000000,0.000000,0.942529,0.057471,0.000000,0.000000
1,454817,6,57,-96,-12,57,9.614035,12.000000,4.122807,12.000000,2.473684,67,0.087719,5,6.000000,0.105263,1.000000,0.017544,49.000000,6.000000,2.000000,0.000000,0.859649,0.105263,0.035088,0.000000
2,307337,6,67,-64,-2,67,20.611940,36.000000,13.731343,36.000000,0.776119,18,0.000000,0,3.000000,0.044776,0.000000,0.000000,61.000000,5.000000,1.000000,0.000000,0.910448,0.074627,0.014925,0.000000
3,183913,6,67,-96,-3,67,11.477612,18.000000,5.761194,18.000000,0.000000,0,0.000000,0,0.000000,0.000000,0.000000,0.000000,62.000000,5.000000,0.000000,0.000000,0.925373,0.074627,0.000000,0.000000
4,112560,3,35,-72,-29,35,12.800000,18.000000,7.085714,18.000000,0.000000,0,0.000000,0,0.000000,0.000000,0.000000,0.000000,33.000000,2.000000,0.000000,0.000000,0.942857,0.057143,0.000000,0.000000


In [7]:
con.execute(f"""
COPY (
    WITH src AS (
        SELECT
            SK_ID_CURR,
            SK_ID_PREV,
            MONTHS_BALANCE,
            AMT_BALANCE,
            AMT_CREDIT_LIMIT_ACTUAL,
            AMT_DRAWINGS_CURRENT,
            AMT_PAYMENT_CURRENT,
            AMT_PAYMENT_TOTAL_CURRENT,
            AMT_INST_MIN_REGULARITY,
            CNT_DRAWINGS_CURRENT,
            CNT_DRAWINGS_ATM_CURRENT,
            CNT_DRAWINGS_POS_CURRENT,
            CNT_INSTALMENT_MATURE_CUM,
            SK_DPD,
            SK_DPD_DEF,
            NAME_CONTRACT_STATUS,

            CASE
                WHEN AMT_CREDIT_LIMIT_ACTUAL IS NULL OR AMT_CREDIT_LIMIT_ACTUAL = 0 THEN NULL
                ELSE AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL
            END AS CC_BALANCE_LIMIT_RATIO,

            CASE
                WHEN AMT_CREDIT_LIMIT_ACTUAL IS NULL OR AMT_CREDIT_LIMIT_ACTUAL = 0 THEN NULL
                ELSE AMT_DRAWINGS_CURRENT / AMT_CREDIT_LIMIT_ACTUAL
            END AS CC_DRAWINGS_LIMIT_RATIO,

            CASE
                WHEN AMT_INST_MIN_REGULARITY IS NULL OR AMT_INST_MIN_REGULARITY = 0 THEN NULL
                ELSE AMT_PAYMENT_CURRENT / AMT_INST_MIN_REGULARITY
            END AS CC_PAYMENT_MIN_RATIO,

            CASE
                WHEN AMT_BALANCE IS NULL OR AMT_BALANCE = 0 THEN NULL
                ELSE AMT_PAYMENT_TOTAL_CURRENT / AMT_BALANCE
            END AS CC_TOTAL_PAYMENT_BALANCE_RATIO,

            CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END AS CC_IS_DPD,
            CASE WHEN SK_DPD_DEF > 0 THEN 1 ELSE 0 END AS CC_IS_DPD_DEF,
            CASE WHEN AMT_BALANCE > 0 THEN 1 ELSE 0 END AS CC_BALANCE_POSITIVE,
            CASE WHEN AMT_PAYMENT_CURRENT < AMT_INST_MIN_REGULARITY THEN 1 ELSE 0 END AS CC_PAYMENT_LESS_THAN_MIN
        FROM read_csv_auto('{cc_path}', header=true)
    )
    SELECT
        SK_ID_CURR,
        COUNT(DISTINCT SK_ID_PREV) AS CC_SK_ID_PREV_NUNIQUE,
        COUNT(*) AS CC_RECORD_COUNT,

        MIN(MONTHS_BALANCE) AS CC_MONTHS_BALANCE_MIN,
        MAX(MONTHS_BALANCE) AS CC_MONTHS_BALANCE_MAX,
        COUNT(MONTHS_BALANCE) AS CC_MONTHS_BALANCE_SIZE,

        AVG(AMT_BALANCE) AS CC_AMT_BALANCE_MEAN,
        MAX(AMT_BALANCE) AS CC_AMT_BALANCE_MAX,
        SUM(AMT_BALANCE) AS CC_AMT_BALANCE_SUM,

        AVG(AMT_CREDIT_LIMIT_ACTUAL) AS CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,
        MAX(AMT_CREDIT_LIMIT_ACTUAL) AS CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,

        AVG(CC_BALANCE_LIMIT_RATIO) AS CC_BALANCE_LIMIT_RATIO_MEAN,
        MAX(CC_BALANCE_LIMIT_RATIO) AS CC_BALANCE_LIMIT_RATIO_MAX,

        AVG(AMT_DRAWINGS_CURRENT) AS CC_AMT_DRAWINGS_CURRENT_MEAN,
        MAX(AMT_DRAWINGS_CURRENT) AS CC_AMT_DRAWINGS_CURRENT_MAX,
        SUM(AMT_DRAWINGS_CURRENT) AS CC_AMT_DRAWINGS_CURRENT_SUM,

        AVG(CC_DRAWINGS_LIMIT_RATIO) AS CC_DRAWINGS_LIMIT_RATIO_MEAN,
        MAX(CC_DRAWINGS_LIMIT_RATIO) AS CC_DRAWINGS_LIMIT_RATIO_MAX,

        AVG(AMT_PAYMENT_CURRENT) AS CC_AMT_PAYMENT_CURRENT_MEAN,
        MAX(AMT_PAYMENT_CURRENT) AS CC_AMT_PAYMENT_CURRENT_MAX,
        SUM(AMT_PAYMENT_CURRENT) AS CC_AMT_PAYMENT_CURRENT_SUM,

        AVG(AMT_PAYMENT_TOTAL_CURRENT) AS CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN,
        MAX(AMT_PAYMENT_TOTAL_CURRENT) AS CC_AMT_PAYMENT_TOTAL_CURRENT_MAX,
        SUM(AMT_PAYMENT_TOTAL_CURRENT) AS CC_AMT_PAYMENT_TOTAL_CURRENT_SUM,

        AVG(AMT_INST_MIN_REGULARITY) AS CC_AMT_INST_MIN_REGULARITY_MEAN,
        MAX(AMT_INST_MIN_REGULARITY) AS CC_AMT_INST_MIN_REGULARITY_MAX,

        AVG(CC_PAYMENT_MIN_RATIO) AS CC_PAYMENT_MIN_RATIO_MEAN,
        MAX(CC_PAYMENT_MIN_RATIO) AS CC_PAYMENT_MIN_RATIO_MAX,

        AVG(CC_TOTAL_PAYMENT_BALANCE_RATIO) AS CC_TOTAL_PAYMENT_BALANCE_RATIO_MEAN,
        MAX(CC_TOTAL_PAYMENT_BALANCE_RATIO) AS CC_TOTAL_PAYMENT_BALANCE_RATIO_MAX,

        AVG(CNT_DRAWINGS_CURRENT) AS CC_CNT_DRAWINGS_CURRENT_MEAN,
        MAX(CNT_DRAWINGS_CURRENT) AS CC_CNT_DRAWINGS_CURRENT_MAX,
        AVG(CNT_DRAWINGS_ATM_CURRENT) AS CC_CNT_DRAWINGS_ATM_CURRENT_MEAN,
        AVG(CNT_DRAWINGS_POS_CURRENT) AS CC_CNT_DRAWINGS_POS_CURRENT_MEAN,

        AVG(CNT_INSTALMENT_MATURE_CUM) AS CC_CNT_INSTALMENT_MATURE_CUM_MEAN,
        MAX(CNT_INSTALMENT_MATURE_CUM) AS CC_CNT_INSTALMENT_MATURE_CUM_MAX,

        AVG(SK_DPD) AS CC_SK_DPD_MEAN,
        MAX(SK_DPD) AS CC_SK_DPD_MAX,
        AVG(SK_DPD_DEF) AS CC_SK_DPD_DEF_MEAN,
        MAX(SK_DPD_DEF) AS CC_SK_DPD_DEF_MAX,

        SUM(CC_IS_DPD) AS CC_DPD_COUNT,
        AVG(CC_IS_DPD) AS CC_DPD_RATIO,
        SUM(CC_IS_DPD_DEF) AS CC_DPD_DEF_COUNT,
        AVG(CC_IS_DPD_DEF) AS CC_DPD_DEF_RATIO,

        SUM(CC_BALANCE_POSITIVE) AS CC_BALANCE_POSITIVE_COUNT,
        AVG(CC_BALANCE_POSITIVE) AS CC_BALANCE_POSITIVE_RATIO,

        SUM(CC_PAYMENT_LESS_THAN_MIN) AS CC_PAYMENT_LESS_THAN_MIN_COUNT,
        AVG(CC_PAYMENT_LESS_THAN_MIN) AS CC_PAYMENT_LESS_THAN_MIN_RATIO,

        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Active' THEN 1 ELSE 0 END) AS CC_STATUS_ACTIVE_COUNT,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Completed' THEN 1 ELSE 0 END) AS CC_STATUS_COMPLETED_COUNT,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Demand' THEN 1 ELSE 0 END) AS CC_STATUS_DEMAND_COUNT,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Signed' THEN 1 ELSE 0 END) AS CC_STATUS_SIGNED_COUNT,

        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Active' THEN 1.0 ELSE 0.0 END) AS CC_STATUS_ACTIVE_RATIO,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Completed' THEN 1.0 ELSE 0.0 END) AS CC_STATUS_COMPLETED_RATIO,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Demand' THEN 1.0 ELSE 0.0 END) AS CC_STATUS_DEMAND_RATIO,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Signed' THEN 1.0 ELSE 0.0 END) AS CC_STATUS_SIGNED_RATIO
    FROM src
    GROUP BY SK_ID_CURR
) TO '{(PROCESSED_DATA_DIR / "credit_card_features.parquet").as_posix()}'
(FORMAT PARQUET);
""")

print("Saved credit_card_features.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved credit_card_features.parquet


In [8]:
cc_features = pd.read_parquet(PROCESSED_DATA_DIR / "credit_card_features.parquet")
print("Credit card features shape:", cc_features.shape)
display(cc_features.head())

Credit card features shape: (103558, 56)


,SK_ID_CURR,CC_SK_ID_PREV_NUNIQUE,CC_RECORD_COUNT,CC_MONTHS_BALANCE_MIN,CC_MONTHS_BALANCE_MAX,CC_MONTHS_BALANCE_SIZE,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_AMT_BALANCE_SUM,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_BALANCE_LIMIT_RATIO_MEAN,CC_BALANCE_LIMIT_RATIO_MAX,CC_AMT_DRAWINGS_CURRENT_MEAN,CC_AMT_DRAWINGS_CURRENT_MAX,CC_AMT_DRAWINGS_CURRENT_SUM,CC_DRAWINGS_LIMIT_RATIO_MEAN,CC_DRAWINGS_LIMIT_RATIO_MAX,CC_AMT_PAYMENT_CURRENT_MEAN,CC_AMT_PAYMENT_CURRENT_MAX,CC_AMT_PAYMENT_CURRENT_SUM,CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN,CC_AMT_PAYMENT_TOTAL_CURRENT_MAX,CC_AMT_PAYMENT_TOTAL_CURRENT_SUM,CC_AMT_INST_MIN_REGULARITY_MEAN,CC_AMT_INST_MIN_REGULARITY_MAX,CC_PAYMENT_MIN_RATIO_MEAN,CC_PAYMENT_MIN_RATIO_MAX,CC_TOTAL_PAYMENT_BALANCE_RATIO_MEAN,CC_TOTAL_PAYMENT_BALANCE_RATIO_MAX,CC_CNT_DRAWINGS_CURRENT_MEAN,CC_CNT_DRAWINGS_CURRENT_MAX,CC_CNT_DRAWINGS_ATM_CURRENT_MEAN,CC_CNT_DRAWINGS_POS_CURRENT_MEAN,CC_CNT_INSTALMENT_MATURE_CUM_MEAN,CC_CNT_INSTALMENT_MATURE_CUM_MAX,CC_SK_DPD_MEAN,CC_SK_DPD_MAX,CC_SK_DPD_DEF_MEAN,CC_SK_DPD_DEF_MAX,CC_DPD_COUNT,CC_DPD_RATIO,CC_DPD_DEF_COUNT,CC_DPD_DEF_RATIO,CC_BALANCE_POSITIVE_COUNT,CC_BALANCE_POSITIVE_RATIO,CC_PAYMENT_LESS_THAN_MIN_COUNT,CC_PAYMENT_LESS_THAN_MIN_RATIO,CC_STATUS_ACTIVE_COUNT,CC_STATUS_COMPLETED_COUNT,CC_STATUS_DEMAND_COUNT,CC_STATUS_SIGNED_COUNT,CC_STATUS_ACTIVE_RATIO,CC_STATUS_COMPLETED_RATIO,CC_STATUS_DEMAND_RATIO,CC_STATUS_SIGNED_RATIO
0,205087,1,31,-32,-2,31,"33,941.543226","184,911.480000","1,052,187.840000","162,580.645161",180000,0.191650,1.027286,"5,804.129032","179,928.000000","179,928.000000",0.032245,0.999600,"17,043.991071","22,500.000000","238,615.875000","7,697.286290","22,500.000000","238,615.875000","2,150.483400","9,351.225000",25.078029,222.466667,114.250242,"1,253.333333",0.032258,1,0.000000,0.000000,3.960000,11.000000,0.225806,7,0.225806,7,1.000000,0.032258,1.000000,0.032258,11.000000,0.354839,0.000000,0.000000,31.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
1,328431,1,60,-61,-2,60,0.000000,0.000000,0.000000,"78,750.000000",90000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0,NaN,NaN,0.000000,0.000000,0.000000,0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,60.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2,326876,1,15,-15,-1,15,0.000000,0.000000,0.000000,"675,000.000000",675000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0,NaN,NaN,0.000000,0.000000,0.000000,0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,15.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
3,147249,1,96,-96,-1,96,"9,140.360625","89,298.675000","877,474.620000","81,093.750000",90000,0.101560,0.992208,468.750000,"22,500.000000","45,000.000000",0.005208,0.250000,"2,434.478906","45,000.000000","233,709.975000","1,821.796875","45,000.000000","174,892.500000","1,071.104062","5,400.000000",2.403221,12.286000,3.613335,63.884157,0.020833,1,0.020833,0.000000,34.708333,37.000000,0.000000,0,0.000000,0,0.000000,0.000000,0.000000,0.000000,19.000000,0.197917,0.000000,0.000000,96.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
4,296309,1,95,-95,-1,95,"9,326.765368","69,563.385000","886,042.710000","22,736.842105",67500,0.410205,1.030569,"1,710.000000","67,500.000000","162,450.000000",0.075208,1.000000,"2,161.035000","45,000.000000","205,298.325000","2,147.875105","45,000.000000","204,048.135000",819.947967,"3,375.000000",2.602466,13.333333,7.752233,93.882227,0.105263,3,0.108696,0.000000,20.637363,24.000000,0.000000,0,0.000000,0,0.000000,0.000000,0.000000,0.000000,24.000000,0.252632,0.000000,0.000000,95.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000


In [9]:
payment_history_features = (
    base
    .merge(installments_features, on=ID_COL, how="left")
    .merge(pos_features, on=ID_COL, how="left")
    .merge(cc_features, on=ID_COL, how="left")
)

print("Combined payment history features shape:", payment_history_features.shape)
print("Unique key:", payment_history_features[ID_COL].is_unique)

display(payment_history_features.head())

Combined payment history features shape: (356255, 115)
Unique key: True


,SK_ID_CURR,INS_SK_ID_PREV_NUNIQUE,INS_RECORD_COUNT,INS_NUM_INSTALMENT_VERSION_NUNIQUE,INS_NUM_INSTALMENT_NUMBER_MAX,INS_DPD_MEAN,INS_DPD_MAX,INS_DPD_SUM,INS_DBD_MEAN,INS_DBD_MAX,INS_DBD_SUM,INS_LATE_COUNT,INS_LATE_RATIO,INS_EARLY_COUNT,INS_EARLY_RATIO,INS_UNDERPAID_COUNT,INS_UNDERPAID_RATIO,INS_OVERPAID_COUNT,INS_OVERPAID_RATIO,INS_PAYMENT_RATIO_MEAN,INS_PAYMENT_RATIO_MIN,INS_PAYMENT_RATIO_MAX,INS_PAYMENT_DIFF_MEAN,INS_PAYMENT_DIFF_MIN,INS_PAYMENT_DIFF_MAX,INS_PAYMENT_DIFF_SUM,INS_AMT_PAYMENT_SUM,INS_AMT_PAYMENT_MEAN,INS_AMT_INSTALMENT_SUM,INS_AMT_INSTALMENT_MEAN,INS_DAYS_INSTALMENT_MIN,INS_DAYS_INSTALMENT_MAX,INS_DAYS_ENTRY_PAYMENT_MIN,INS_DAYS_ENTRY_PAYMENT_MAX,INS_TOTAL_PAYMENT_TO_INSTALMENT_RATIO,POS_SK_ID_PREV_NUNIQUE,POS_RECORD_COUNT,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_MONTHS_BALANCE_SIZE,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX,POS_DPD_COUNT,POS_DPD_RATIO,POS_DPD_DEF_COUNT,POS_DPD_DEF_RATIO,POS_STATUS_ACTIVE_COUNT,POS_STATUS_COMPLETED_COUNT,POS_STATUS_SIGNED_COUNT,POS_STATUS_DEMAND_COUNT,POS_STATUS_ACTIVE_RATIO,POS_STATUS_COMPLETED_RATIO,POS_STATUS_SIGNED_RATIO,POS_STATUS_DEMAND_RATIO,CC_SK_ID_PREV_NUNIQUE,CC_RECORD_COUNT,CC_MONTHS_BALANCE_MIN,CC_MONTHS_BALANCE_MAX,CC_MONTHS_BALANCE_SIZE,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_AMT_BALANCE_SUM,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_BALANCE_LIMIT_RATIO_MEAN,CC_BALANCE_LIMIT_RATIO_MAX,CC_AMT_DRAWINGS_CURRENT_MEAN,CC_AMT_DRAWINGS_CURRENT_MAX,CC_AMT_DRAWINGS_CURRENT_SUM,CC_DRAWINGS_LIMIT_RATIO_MEAN,CC_DRAWINGS_LIMIT_RATIO_MAX,CC_AMT_PAYMENT_CURRENT_MEAN,CC_AMT_PAYMENT_CURRENT_MAX,CC_AMT_PAYMENT_CURRENT_SUM,CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN,CC_AMT_PAYMENT_TOTAL_CURRENT_MAX,CC_AMT_PAYMENT_TOTAL_CURRENT_SUM,CC_AMT_INST_MIN_REGULARITY_MEAN,CC_AMT_INST_MIN_REGULARITY_MAX,CC_PAYMENT_MIN_RATIO_MEAN,CC_PAYMENT_MIN_RATIO_MAX,CC_TOTAL_PAYMENT_BALANCE_RATIO_MEAN,CC_TOTAL_PAYMENT_BALANCE_RATIO_MAX,CC_CNT_DRAWINGS_CURRENT_MEAN,CC_CNT_DRAWINGS_CURRENT_MAX,CC_CNT_DRAWINGS_ATM_CURRENT_MEAN,CC_CNT_DRAWINGS_POS_CURRENT_MEAN,CC_CNT_INSTALMENT_MATURE_CUM_MEAN,CC_CNT_INSTALMENT_MATURE_CUM_MAX,CC_SK_DPD_MEAN,CC_SK_DPD_MAX,CC_SK_DPD_DEF_MEAN,CC_SK_DPD_DEF_MAX,CC_DPD_COUNT,CC_DPD_RATIO,CC_DPD_DEF_COUNT,CC_DPD_DEF_RATIO,CC_BALANCE_POSITIVE_COUNT,CC_BALANCE_POSITIVE_RATIO,CC_PAYMENT_LESS_THAN_MIN_COUNT,CC_PAYMENT_LESS_THAN_MIN_RATIO,CC_STATUS_ACTIVE_COUNT,CC_STATUS_COMPLETED_COUNT,CC_STATUS_DEMAND_COUNT,CC_STATUS_SIGNED_COUNT,CC_STATUS_ACTIVE_RATIO,CC_STATUS_COMPLETED_RATIO,CC_STATUS_DEMAND_RATIO,CC_STATUS_SIGNED_RATIO
0,100002,1.000000,19.000000,2.000000,19.000000,0.000000,0.000000,0.000000,20.421053,31.000000,388.000000,0.000000,0.000000,19.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,"219,625.695000","11,559.247105","219,625.695000","11,559.247105",-565.000000,-25.000000,-587.000000,-49.000000,1.000000,1.000000,19.000000,-19.000000,-1.000000,19.000000,24.000000,24.000000,15.000000,24.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,19.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,3.000000,25.000000,2.000000,12.000000,0.000000,0.000000,0.000000,7.160000,14.000000,179.000000,0.000000,0.000000,25.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,"1,618,864.650000","64,754.586000","1,618,864.650000","64,754.586000","-2,310.000000",-536.000000,"-2,324.000000",-544.000000,1.000000,3.000000,28.000000,-77.000000,-18.000000,28.000000,10.107143,12.000000,5.785714,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.

In [10]:
payment_history_features.to_parquet(
    PROCESSED_DATA_DIR / "payment_history_features.parquet",
    index=False
)

print("Saved:", PROCESSED_DATA_DIR / "payment_history_features.parquet")

Saved: C:\Coding\Home-Credit-Default-Risk\data\processed\payment_history_features.parquet


In [11]:
feature_cols = [c for c in payment_history_features.columns if c != ID_COL]

print("Number of payment-history features:", len(feature_cols))
print()

display(
    payment_history_features[feature_cols]
    .isnull()
    .mean()
    .sort_values(ascending=False)
    .head(30)
)

Number of payment-history features: 114



CC_PAYMENT_MIN_RATIO_MAX              0.803837
CC_PAYMENT_MIN_RATIO_MEAN             0.803837
CC_TOTAL_PAYMENT_BALANCE_RATIO_MAX    0.802855
CC_TOTAL_PAYMENT_BALANCE_RATIO_MEAN   0.802855
CC_AMT_PAYMENT_CURRENT_SUM            0.797561
CC_AMT_PAYMENT_CURRENT_MEAN           0.797561
CC_AMT_PAYMENT_CURRENT_MAX            0.797561
CC_CNT_DRAWINGS_ATM_CURRENT_MEAN      0.797353
CC_CNT_DRAWINGS_POS_CURRENT_MEAN      0.797353
CC_DRAWINGS_LIMIT_RATIO_MAX           0.712439
CC_BALANCE_LIMIT_RATIO_MEAN           0.712439
CC_BALANCE_LIMIT_RATIO_MAX            0.712439
CC_DRAWINGS_LIMIT_RATIO_MEAN          0.712439
CC_MONTHS_BALANCE_MIN                 0.709315
CC_AMT_INST_MIN_REGULARITY_MAX        0.709315
CC_AMT_INST_MIN_REGULARITY_MEAN       0.709315
CC_AMT_PAYMENT_TOTAL_CURRENT_SUM      0.709315
CC_AMT_PAYMENT_TOTAL_CURRENT_MAX      0.709315
CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN     0.709315
CC_MONTHS_BALANCE_SIZE                0.709315
CC_AMT_BALANCE_MEAN                   0.709315
CC_MONTHS_BAL

In [12]:
check_cols = [
    "INS_RECORD_COUNT",
    "INS_LATE_RATIO",
    "INS_UNDERPAID_RATIO",
    "INS_TOTAL_PAYMENT_TO_INSTALMENT_RATIO",
    "POS_RECORD_COUNT",
    "POS_DPD_RATIO",
    "POS_DPD_DEF_RATIO",
    "CC_RECORD_COUNT",
    "CC_BALANCE_LIMIT_RATIO_MEAN",
    "CC_PAYMENT_MIN_RATIO_MEAN",
    "CC_DPD_RATIO",
]

existing_check_cols = [c for c in check_cols if c in payment_history_features.columns]
display(payment_history_features[existing_check_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
INS_RECORD_COUNT,"339,587.000000",40.064552,41.053343,1.000000,12.000000,25.000000,51.000000,372.000000
INS_LATE_RATIO,"339,587.000000",0.074385,0.114490,0.000000,0.000000,0.017857,0.109375,1.000000
INS_UNDERPAID_RATIO,"339,587.000000",0.082631,0.148284,0.000000,0.000000,0.000000,0.105263,1.000000
INS_TOTAL_PAYMENT_TO_INSTALMENT_RATIO,"339,575.000000",0.993408,0.136955,0.133538,0.962445,1.000000,1.000000,5.485141
POS_RECORD_COUNT,"337,252.000000",29.655445,24.531971,1.000000,12.000000,22.000000,39.000000,295.000000
POS_DPD_RATIO,"337,252.000000",0.021046,0.078553,0.000000,0.000000,0.000000,0.000000,1.000000
POS_DPD_DEF_RATIO,"337,252.000000",0.009903,0.039230,0.000000,0.000000,0.000000,0.000000,1.000000
CC_RECORD_COUNT,"103,558.000000",37.083683,33.483627,1.000000,10.000000,22.000000,75.000000,192.000000
CC_BALANCE_LIMIT_RATIO_MEAN,"102,445.000000",0.320573,0.324382,-0.084848,0.000000,0.241313,0.583274,2.138790
CC_PAYMENT_MIN_RATIO_MEAN,"69,884.000000",18.715104,"1,886.981209",0.000000,1.411504,2.398907,5.010565,"496,093.080083"
